# Partially runnable against the public dataset

**Runs as published:** the exclusion-count tables, and the task-duration analysis
(`Ztable_dur_combined.csv` carries no demographics).

**Does not run:** the attrition GLM, which regresses inclusion on age, gender,
income, religion, education, vote, disability, mental health and chatbot use. Those
columns are removed from the `*_exclusions.csv` files in the public release (see
the Data section of the README), which retain only `iscontrol` and `include`.

Re-running the attrition GLM requires controlled access to the underlying data.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpyro
import numpyro.distributions as dist
from numpyro import sample
from numpyro.infer import MCMC, NUTS, init_to_median
from numpyro.infer.reparam import TransformReparam
from numpyro.handlers import reparam
from numpyro.distributions import transforms
from scipy import stats as scipy_stats
from scipy import stats

import jax
import jax.numpy as jnp
import jax.random as random

from numpyro.infer.initialization import init_to_feasible
from numpyro.infer import Predictive

from itertools import combinations

from statsmodels.stats.outliers_influence import variance_inflation_factor

import arviz as az

# enable multiple MCMC chains in parallel
numpyro.set_host_device_count(4)

# set random seed
rng_key = jax.random.PRNGKey(1)

num_chains = 4

In [ ]:
print(jax.local_device_count())

#get GPU if available
device = jax.devices("gpu")[0] if jax.extend.backend.get_backend().platform == "gpu" else "cpu"
print(device)

# Attrition analyses

In [ ]:
#attrition results

data = {
    "GPT4o": {
        "overall": {"included": 382, "total": 410},
        "test": {"included": 190, "total": 200},
        "control": {"included": 192, "total": 210}
    },
    "mistral": {
        "overall": {"included": 364, "total": 422},
        "test": {"included": 183, "total": 211},
        "control": {"included": 181, "total": 211}
    },
    "claude": {
        "overall": {"included": 401, "total": 420},
        "test": {"included": 190, "total": 197},
        "control": {"included": 211, "total": 223}
    },
    "GPT4o_sycophancy_both": {
        "overall": {"included": 977, "total": 1025},
        "test": {"included": 489, "total": 509},
        "control": {"included": 488, "total": 516}
    },
    "GPT4o_persuasion": {
        "overall": {"included": 734, "total": 783},
        "test": {"included": 379, "total": 403},
        "control": {"included": 355, "total": 380}
    }
}

# Compute percentages for each group and model
print("Model\t\t\tOverall Excl%\tTest Excl%\tControl Excl%")
print("-" * 70)

# Overall totals
total_included = 0
total_participants = 0
total_test_included = 0
total_test_participants = 0 
total_control_included = 0
total_control_participants = 0

for model, stats in data.items():
    # Calculate exclusion percentages
    overall_excl_pct = 100 * (1 - stats["overall"]["included"] / stats["overall"]["total"])
    test_excl_pct = 100 * (1 - stats["test"]["included"] / stats["test"]["total"])
    control_excl_pct = 100 * (1 - stats["control"]["included"] / stats["control"]["total"])
    
    # Update totals
    total_included += stats["overall"]["included"]
    total_participants += stats["overall"]["total"]
    total_test_included += stats["test"]["included"]
    total_test_participants += stats["test"]["total"]
    total_control_included += stats["control"]["included"]
    total_control_participants += stats["control"]["total"]
    
    # Format model name for better display
    model_display = model
    if len(model) < 16:
        model_display = model + "\t"
    if len(model) < 8:
        model_display = model + "\t"
    
    # Print results
    print(f"{model_display}\t{overall_excl_pct:.2f}%\t\t{test_excl_pct:.2f}%\t\t{control_excl_pct:.2f}%")

# Calculate overall totals
overall_excl_pct = 100 * (1 - total_included / total_participants)
test_excl_pct = 100 * (1 - total_test_included / total_test_participants) 
control_excl_pct = 100 * (1 - total_control_included / total_control_participants)

print("-" * 70)
print(f"OVERALL\t\t\t{overall_excl_pct:.2f}%\t\t{test_excl_pct:.2f}%\t\t{control_excl_pct:.2f}%")

# Calculate the difference between test and control exclusion rates
print("\nDifference in exclusion rates (Test - Control):")
print("-" * 50)
for model, stats in data.items():
    test_excl_pct = 100 * (1 - stats["test"]["included"] / stats["test"]["total"])
    control_excl_pct = 100 * (1 - stats["control"]["included"] / stats["control"]["total"])
    diff = test_excl_pct - control_excl_pct
    
    # Format model name for better display
    model_display = model
    if len(model) < 16:
        model_display = model + "\t"
    if len(model) < 8:
        model_display = model + "\t"
    
    print(f"{model_display}\t{diff:.2f}%")

# Overall difference
overall_diff = test_excl_pct - control_excl_pct
print("-" * 50)
print(f"OVERALL\t\t\t{overall_diff:.2f}%")



# 1. Overall test using combined data
total_test_excluded = 0
total_test_total = 0
total_control_excluded = 0
total_control_total = 0

for model, model_data in data.items():
    total_test_excluded += (model_data["test"]["total"] - model_data["test"]["included"])
    total_test_total += model_data["test"]["total"]
    total_control_excluded += (model_data["control"]["total"] - model_data["control"]["included"])
    total_control_total += model_data["control"]["total"]

# Calculate proportions
test_prop = total_test_excluded / total_test_total
control_prop = total_control_excluded / total_control_total

# Two-proportion z-test (more appropriate than t-test for proportions)
def prop_z_test(p1, n1, p2, n2):
    # Pooled proportion
    p_pooled = (p1 * n1 + p2 * n2) / (n1 + n2)
    # Standard error
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    # Z-statistic
    z = (p1 - p2) / se
    # P-value (two-tailed)
    p_value = 2 * (1 - scipy_stats.norm.cdf(abs(z)))  # Fixed here
    return z, p_value

z_stat, p_value = prop_z_test(test_prop, total_test_total, control_prop, total_control_total)

print("\nOverall Comparison of Exclusion Rates:")
print(f"Test group: {test_prop*100:.2f}% excluded ({total_test_excluded} out of {total_test_total})")
print(f"Control group: {control_prop*100:.2f}% excluded ({total_control_excluded} out of {total_control_total})")
print(f"Difference: {(test_prop-control_prop)*100:.2f}%")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"{'Statistically significant' if p_value < 0.05 else 'Not statistically significant'} at α = 0.05")

# 2. Individual tests for each model
print("\nIndividual Model Comparisons:")
print("-" * 80)
print("Model\t\t\tTest Excl%\tControl Excl%\tDiff\t\tZ-stat\t\tP-value\tSignificant")
print("-" * 80)

for model, model_data in data.items():
    test_excluded = model_data["test"]["total"] - model_data["test"]["included"]
    test_total = model_data["test"]["total"]
    control_excluded = model_data["control"]["total"] - model_data["control"]["included"]
    control_total = model_data["control"]["total"]
    
    test_prop = test_excluded / test_total
    control_prop = control_excluded / control_total
    
    # Calculate z-test
    z_stat, p_value = prop_z_test(test_prop, test_total, control_prop, control_total)
    
    # Format model name for better display
    model_display = model
    if len(model) < 16:
        model_display = model + "\t"
    if len(model) < 8:
        model_display = model + "\t"
    
    # Print results
    print(f"{model_display}\t{test_prop*100:.2f}%\t\t{control_prop*100:.2f}%\t\t{(test_prop-control_prop)*100:.2f}%\t\t{z_stat:.4f}\t\t{p_value:.4f}\t{'*' if p_value < 0.05 else ''}")

# Selective attrition tests

In [ ]:
import statsmodels.api as sm

# Load all datasets
files = {
    'GPT4o': '../data/GPT4o_exclusions.csv',
    'GPT4o_persuasion': '../data/GPT4o_persuasion_exclusions.csv',
    'GPT4o_sycophancy': '../data/GPT4o_sycophancy_exclusions.csv',
    'mistral': '../data/mistral_exclusions.csv',
    'claude': '../data/claude_exclusions.csv'
}

progressive = ['labour', 'green', 'liberal', 'SNP', 'sinn_fein', 'plaid_cymru']
conservative = ['conservative', 'reform_UK', 'unionist']

dfs = []
for name, filepath in files.items():
    df = pd.read_csv(filepath)
    df['study'] = name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# Create binary variables
df_all['vote_binary'] = np.where(
    df_all['vote'].isin(progressive), 0.5,
    np.where(df_all['vote'].isin(conservative), -0.5, np.nan))

df_all['gender_binary'] = np.where(
    df_all['gender'] == 'male', 0.5,
    np.where(df_all['gender'] == 'female', -0.5, np.nan))

df_all['religion_binary'] = np.where(
    df_all['religion'] == 'no_religion', -0.5,
    np.where(df_all['religion'].isin(['prefer_not_to_say', 'missing']), np.nan, 0.5))

df_all['mental_health_binary'] = np.where(
    df_all['mental_health'] == 'yes', 0.5,
    np.where(df_all['mental_health'] == 'no', -0.5, np.nan))

df_all['disabled_binary'] = np.where(
    df_all['disabled'].isin(['yes_disabled', 'yes_minor', 'yes_not_registered']), 0.5,
    np.where(df_all['disabled'] == 'no', -0.5, np.nan))

covariates = ['age', 'gender_binary', 'income', 'religion_binary',
              'education', 'vote_binary', 'disabled_binary', 
              'mental_health_binary', 'chatbot_use']

df_clean = df_all.dropna(subset=['include', 'iscontrol'] + covariates).copy()

print(f"N = {len(df_clean)}, Included = {int(df_clean['include'].sum())}, "
      f"Excluded = {int((1 - df_clean['include']).sum())}")
print(f"Attrition rate: {(1 - df_clean['include']).mean():.1%}")

# Encode
categorical_cols = ['age', 'income', 'education', 'chatbot_use']
numeric_cols = ['gender_binary', 'religion_binary', 'vote_binary',
                'disabled_binary', 'mental_health_binary']


print("Categorical variable levels used in the model:")
print(f"{'='*50}")
for col in categorical_cols:
    levels = sorted(df_clean[col].dropna().unique())
    print(f"\n{col} ({len(levels)} levels):")
    for level in levels:
        print(f"  - {level}")

print("Numerical variable distributions:")
print(f"{'='*50}")
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  {df_clean[col].value_counts().to_dict()}")
    print(f"  Mean = {df_clean[col].mean():.3f}")

df_cat = pd.get_dummies(df_clean[categorical_cols], drop_first=True).astype(float)
df_num = df_clean[numeric_cols].astype(float)
df_encoded = pd.concat([df_num, df_cat], axis=1)

# Restricted model
X_restricted = df_encoded.copy()
X_restricted['iscontrol'] = df_clean['iscontrol'].astype(float).values
X_restricted = sm.add_constant(X_restricted)

# Full model with interactions
X_full = df_encoded.copy()
X_full['iscontrol'] = df_clean['iscontrol'].astype(float).values
for col in df_encoded.columns:
    X_full[f'iscontrol_x_{col}'] = X_full['iscontrol'] * X_full[col]
X_full = sm.add_constant(X_full)

y = df_clean['include'].astype(float).values

model_restricted = sm.OLS(y, X_restricted).fit()
model_full = sm.OLS(y, X_full).fit()

f_test = model_full.compare_f_test(model_restricted)

print(f"\nJoint F-test of treatment × covariate interactions:")
print(f"  F({int(f_test[2])}, {int(model_full.df_resid)}) = {f_test[0]:.3f}")
print(f"  p-value = {f_test[1]:.4f}")

if f_test[1] > 0.05:
    print("  → No evidence of selective attrition")
else:
    print("  → Evidence of selective attrition!")

# Manipulation check analyses

In [ ]:
# Load data
df_syc = pd.read_csv('../data/GPT4o_sycophancy_model_reliable.csv')
df_per = pd.read_csv('../data/GPT4o_persuasion_model_reliable.csv')

# Add condition labels
df_syc['condition'] = df_syc['iscontrol'].map({0: 'control', 1: 'sycophancy'})
df_per['condition'] = df_per['iscontrol'].map({0: 'control', 1: 'persuasion'})


# Check the range of values
print("SYCOPHANCY DATA")
for var in ['model_reliable', 'model_agree']:
    print(f"  {var}: min = {df_syc[var].min()}, max = {df_syc[var].max()}, "
          f"unique = {sorted(df_syc[var].dropna().unique())}")

print("\nPERSUASION DATA")
for var in ['model_reliable', 'model_agree']:
    print(f"  {var}: min = {df_per[var].min()}, max = {df_per[var].max()}, "
          f"unique = {sorted(df_per[var].dropna().unique())}")


print("=" * 70)
print("SYCOPHANCY STUDY")
print("=" * 70)

# Test against 4 for each condition
for condition in ['control', 'sycophancy']:
    for var in ['model_reliable', 'model_agree']:
        subset = df_syc[df_syc['condition'] == condition][var].dropna()
        t_stat, p_value = stats.ttest_1samp(subset, 4)
        print(f"{condition:15s} - {var:15s}: M = {subset.mean():.3f}, "
              f"SD = {subset.std():.3f}, N = {len(subset)}, "
              f"t = {t_stat:.3f}, p = {p_value:.4f}")

# Between conditions
for var in ['model_reliable', 'model_agree']:
    control = df_syc[df_syc['condition'] == 'control'][var].dropna()
    treatment = df_syc[df_syc['condition'] == 'sycophancy'][var].dropna()
    t_stat, p_value = stats.ttest_ind(control, treatment)
    print(f"\nControl vs Sycophancy - {var}: "
          f"M_control = {control.mean():.3f}, M_syc = {treatment.mean():.3f}, "
          f"t = {t_stat:.3f}, p = {p_value:.4f}")

print("\n" + "=" * 70)
print("PERSUASION STUDY")
print("=" * 70)

# Test against 4 for each condition
for condition in ['control', 'persuasion']:
    for var in ['model_reliable', 'model_agree']:
        subset = df_per[df_per['condition'] == condition][var].dropna()
        t_stat, p_value = stats.ttest_1samp(subset, 4)
        print(f"{condition:15s} - {var:15s}: M = {subset.mean():.3f}, "
              f"SD = {subset.std():.3f}, N = {len(subset)}, "
              f"t = {t_stat:.3f}, p = {p_value:.4f}")

# Between conditions
for var in ['model_reliable', 'model_agree']:
    control = df_per[df_per['condition'] == 'control'][var].dropna()
    treatment = df_per[df_per['condition'] == 'persuasion'][var].dropna()
    t_stat, p_value = stats.ttest_ind(control, treatment)
    print(f"\nControl vs Persuasion - {var}: "
          f"M_control = {control.mean():.3f}, M_per = {treatment.mean():.3f}, "
          f"t = {t_stat:.3f}, p = {p_value:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (df, study_name, treat_name) in zip(axes, 
    [(df_syc, 'Sycophancy', 'sycophancy'),
     (df_per, 'Persuasion', 'persuasion')]):
    
    x = np.arange(2)  # reliable, agree
    width = 0.35
    
    for i, (group, group_label, color) in enumerate([
        ('control', 'Control', '#888888'),
        (treat_name, 'Treatment', '#E74C3C' if treat_name == 'sycophancy' else '#3498DB')
    ]):
        means = []
        
        for var in ['model_reliable', 'model_agree']:
            data = df[df['condition'] == group][var].dropna()
            means.append(data.mean())
        
        bars = ax.bar(x + (i - 0.5) * width, means, width,
                      label=group_label, color=color, 
                      edgecolor='black', linewidth=0.5)
        
        # Add mean labels on bars
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                    f'{mean:.2f}', ha='center', va='bottom', fontsize=10)
    
    ax.axhline(y=4, color='black', linestyle='--', linewidth=0.8,
               alpha=0.5, label='Midpoint (4)')
    ax.set_xticks(x)
    ax.set_xticklabels(['Reliable', 'Agree'], fontsize=12)
    ax.set_title(f'{study_name} Study', fontsize=13)
    ax.set_ylim(1, 7)
    ax.set_yticks(range(1, 8))
    ax.legend(fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Mean Rating (1-7)', fontsize=12)

plt.suptitle('Participant Perceptions of AI Model: Control vs. Treatment',
             fontsize=14, y=1.02)
plt.tight_layout()
# plt.savefig('../plots/manipulation_check_barplot.png', dpi=300, bbox_inches='tight')
plt.show()

# Duration analyses


## Data preparation

In [ ]:
#duration data
dur_data = pd.read_csv('../data/Ztable_dur_combined.csv')
print(len(dur_data))
print(dur_data["model"].unique())
dur_data.head(10)

In [ ]:
#exclude data for model = 'GPT4o_sycophancy_both' and 'GPT4o_persuasion'
dur_data = dur_data[~dur_data['model'].isin(['GPT4o_sycophancy_both', 'GPT4o_persuasion'])]

#group dur_data["task_1Seconds"] and dur_data["task_2Seconds"] by searchornot
dur_data["task_1Seconds"] = dur_data["task_1Seconds"].astype(float)
dur_data["task_2Seconds"] = dur_data["task_2Seconds"].astype(float)

grouped_data = dur_data.groupby('iscontrol').agg({
    'task_1Seconds': ['mean', 'median', 'std'],
    'task_2Seconds': ['mean', 'median', 'std']
})

print(grouped_data)

# Compute overall mean across both tasks for each group
dur_data['overall_mean'] = dur_data[['task_1Seconds', 'task_2Seconds']].mean(axis=1)
overall_stats = dur_data.groupby('iscontrol').agg(
    overall_mean_mean=('overall_mean', 'mean'),
    overall_mean_std=('overall_mean', 'std')
)

print("\nOverall Means and Standard Deviations Across Both Tasks:")
print(overall_stats)

In [ ]:
# Set the style for better looking plots
sns.set(style="whitegrid")

# Create a figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Get unique values of searchornot
search_conditions = dur_data['iscontrol'].unique()

# Define colors and labels for better visualization
colors = ['#3498db', '#e74c3c']  # Blue and Red
condition_labels = {condition: f"Condition: {condition}" for condition in search_conditions}

# Plot Task 1 - both conditions in same plot
for i, condition in enumerate(search_conditions):
    # Filter data for this condition
    subset = dur_data[dur_data['iscontrol'] == condition]
    
    # Plot histogram for Task 1
    sns.histplot(subset['task_1Seconds'], kde=True, ax=axes[0], 
                 color=colors[i], bins=10, alpha=0.6,
                 label=condition_labels[condition])
    
    # Add mean line
    mean_val = subset['task_1Seconds'].mean()
    axes[0].axvline(mean_val, color=colors[i], linestyle='--', 
                   alpha=0.8, linewidth=2,
                   label=f'{condition_labels[condition]} Mean: {mean_val:.2f}s')

# Add labels and title for Task 1
axes[0].set_title('Task 1 Duration by Search Condition', fontsize=14)
axes[0].set_xlabel('Time (seconds)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 6000)  # Set x-axis range from 0 to 6000 seconds

# Plot Task 2 - both conditions in same plot
for i, condition in enumerate(search_conditions):
    # Filter data for this condition
    subset = dur_data[dur_data['iscontrol'] == condition]
    
    # Plot histogram for Task 2
    sns.histplot(subset['task_2Seconds'], kde=True, ax=axes[1], 
                 color=colors[i], bins=10, alpha=0.6,
                 label=condition_labels[condition])
    
    # Add mean line
    mean_val = subset['task_2Seconds'].mean()
    axes[1].axvline(mean_val, color=colors[i], linestyle='--', 
                   alpha=0.8, linewidth=2,
                   label=f'{condition_labels[condition]} Mean: {mean_val:.2f}s')
# Add labels and title for Task 2
axes[1].set_title('Task 2 Duration by Search Condition', fontsize=14)
axes[1].set_xlabel('Time (seconds)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, 6000)  # Set x-axis range from 0 to 6000 seconds

# Add an overall title
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for the suptitle

# Save and show the figure
plt.savefig('../plots/task_duration_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#do a t-test for dur_data["task_1Seconds"] and dur_data["task_1Seconds"], comparing dur_data['iscontrol'] == 0 and comparing dur_data['iscontrol'] == 1
from scipy import stats
from scipy.stats import ttest_ind


# Remove NaN values and create clean datasets
dur_data_clean = dur_data.copy()

# Check for NaNs and report
task1_nan_count = dur_data_clean['task_1Seconds'].isna().sum()
task2_nan_count = dur_data_clean['task_2Seconds'].isna().sum()
iscontrol_nan_count = dur_data_clean['iscontrol'].isna().sum()

print(f"NaN counts before cleaning:")
print(f"task_1Seconds: {task1_nan_count}")
print(f"task_2Seconds: {task2_nan_count}")
print(f"iscontrol: {iscontrol_nan_count}")

# Remove NaNs
dur_data_clean = dur_data_clean.dropna(subset=['iscontrol', 'task_1Seconds', 'task_2Seconds'])

print(f"\nOriginal data shape: {dur_data.shape}")
print(f"Clean data shape: {dur_data_clean.shape}")
print(f"Removed {dur_data.shape[0] - dur_data_clean.shape[0]} rows with NaN values")

# Create groups for t-test
control_group = dur_data_clean[dur_data_clean['iscontrol'] == 1]
experimental_group = dur_data_clean[dur_data_clean['iscontrol'] == 0]

print(f"\nGroups after cleaning:")
print(f"Control group (iscontrol=1): {len(control_group)} participants")
print(f"Experimental group (iscontrol=0): {len(experimental_group)} participants")

# T-test for Task 1
t_stat_task1, p_value_task1 = ttest_ind(
    control_group['task_1Seconds'], 
    experimental_group['task_1Seconds'],
    equal_var=False  # Welch's t-test (doesn't assume equal variances)
)

# T-test for Task 2
t_stat_task2, p_value_task2 = ttest_ind(
    control_group['task_2Seconds'], 
    experimental_group['task_2Seconds'],
    equal_var=False  # Welch's t-test (doesn't assume equal variances)
)

# Print results
print("\n==== T-Test Results ====")
print("\nTask 1 Seconds - Control vs Experimental:")
print(f"t-statistic: {t_stat_task1:.4f}")
print(f"p-value: {p_value_task1:.4f}")

print("\nTask 2 Seconds - Control vs Experimental:")
print(f"t-statistic: {t_stat_task2:.4f}")
print(f"p-value: {p_value_task2:.4f}")


In [ ]:
dur_data["subject_id"] = np.arange(len(dur_data))

df1 = dur_data[['subject_id', 'model', 'task_1Seconds', 'iscontrol', 'searchornot']]
df1 = df1.rename(columns={'task_1Seconds': 'search_duration'})

# Second DataFrame for task_2Seconds
df2 = dur_data[['subject_id', 'model', 'task_2Seconds', 'iscontrol', 'searchornot']]
df2 = df2.rename(columns={'task_2Seconds': 'search_duration'})

# Concatenate the two DataFrames
dur_data_combined = pd.concat([df1, df2], ignore_index=True)

print(len(dur_data_combined))
dur_data_combined.head(10)

In [ ]:
# Filter out invalid values from search_durations
valid_mask = (dur_data_combined['search_duration'] > 0) & dur_data_combined['search_duration'].notna() & np.isfinite(dur_data_combined['search_duration'])

# Apply the mask to filter the data
dur_data_combined = dur_data_combined[valid_mask]

# Extract filtered arrays
subject_ids = dur_data_combined['subject_id'].values
search_durations = dur_data_combined['search_duration'].values
iscontrol = dur_data_combined['iscontrol'].values

# Encode subject_id to be used as index
subject_id_unique = np.unique(subject_ids)
subject_id_map = {subject: index for index, subject in enumerate(subject_id_unique)}
subject_id_indices = np.vectorize(subject_id_map.get)(subject_ids)


# Define the model
def hierarchical_model(subject_id_indices, search_durations, iscontrol):
    # Global intercept
    mu_intercept = numpyro.sample('mu_intercept', dist.Normal(0, 1))
    # For iscontrol effect
    mu_iscontrol = numpyro.sample('mu_iscontrol', dist.Normal(0, 1))
    # Random effects for subjects
    sigma_subject = numpyro.sample('sigma_subject', dist.Exponential(1))
    
    # Non-centered parametrization for stability
    subject_intercepts_raw = numpyro.sample('subject_intercepts_raw', dist.Normal(0, 1), sample_shape=(len(subject_id_unique),))
    subject_intercepts = mu_intercept + sigma_subject * subject_intercepts_raw
    
    # Linear model
    mu = mu_intercept + mu_iscontrol * iscontrol + subject_intercepts[subject_id_indices]
    
    # Ensuring positive mean to avoid invalid Gamma distribution parameters
    mu = numpyro.deterministic('mu', jnp.exp(mu))
    
    # Gamma distribution parameters
    alpha = numpyro.sample('alpha', dist.Exponential(1))
    beta = alpha / mu
    
    # Likelihood
    with numpyro.plate('data', len(search_durations)):
        numpyro.sample('obs', dist.Gamma(alpha, beta), obs=search_durations)


# Initialize MCMC parameters
nuts_kernel = NUTS(hierarchical_model)
mcmc = MCMC(nuts_kernel, num_samples=2000, num_warmup=2000, num_chains=num_chains)
rng_key = random.PRNGKey(0)

# Run MCMC
mcmc.run(rng_key, subject_id_indices, search_durations, iscontrol)
mcmc.print_summary()

# Extract samples
samples = mcmc.get_samples()

In [ ]:
# Define the null model without the iscontrol effect
def null_model(subject_id_indices, search_durations):
    # Global intercept
    mu_intercept = numpyro.sample('mu_intercept', dist.Normal(0, 1))
    
    # Random effects for subjects
    sigma_subject = numpyro.sample('sigma_subject', dist.Exponential(1))
    
    # Non-centered parametrization for stability
    subject_intercepts_raw = numpyro.sample('subject_intercepts_raw', dist.Normal(0, 1), sample_shape=(len(subject_id_unique),))
    subject_intercepts = mu_intercept + sigma_subject * subject_intercepts_raw
    
    # Linear model
    mu = mu_intercept + subject_intercepts[subject_id_indices]
    
    # Ensuring positive mean to avoid invalid Gamma distribution parameters
    mu = numpyro.deterministic('mu', jnp.exp(mu))
    
    # Gamma distribution parameters
    alpha = numpyro.sample('alpha', dist.Exponential(1))
    beta = alpha / mu
    
    # Likelihood
    with numpyro.plate('data', len(search_durations)):
        numpyro.sample('obs', dist.Gamma(alpha, beta), obs=search_durations)

# Initialize MCMC parameters
nuts_kernel = NUTS(null_model)
mcmc_null = MCMC(nuts_kernel, num_samples=2000, num_warmup=2000, num_chains=num_chains)
rng_key = random.PRNGKey(0)

# Run MCMC
mcmc_null.run(rng_key, subject_id_indices, search_durations)
mcmc_null.print_summary()

# Extract samples
samples_null = mcmc_null.get_samples()

#### Model comparison

In [ ]:
# compare the two models using WAIC
waic_model_full = az.waic(mcmc)
waic_model_null = az.waic(mcmc_null)

comparison = az.compare({"GLM_full": mcmc, "GLM_null": mcmc_null}, ic="waic")
print(comparison)

#plot the comparison
az.plot_compare(comparison)